# UPS Parcel Dataset 2025: Descriptive Analytics & Visualization
**Goal:** understand where the shipping money goes, what drives cost, and where there are savings opportunities. Everything here is *descriptive* (no predictive modelling).

**Structure**
1. Setup & load (your original cells)
2. Data cleaning & quality check
3. Derived metrics (feature engineering)
4. KPI summary
5. Visualizations: each followed by a printed **derived result**
6. Summary of findings

## 1. Setup & load

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd               # working with data in tables
import numpy as np                # advanced calculations
import matplotlib.pyplot as plt   # chart creation
import matplotlib.ticker as mtick # $ and % axis formatting
import seaborn as sns             # more complex charts (heatmaps, boxplots)

# One consistent, colorblind-safe look for every chart
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
sns.set_theme(style="whitegrid", palette=COLORS)
plt.rcParams.update({"figure.figsize": (11, 5), "axes.titleweight": "bold",
                     "axes.titlelocation": "left", "axes.spines.top": False,
                     "axes.spines.right": False})
usd = mtick.FuncFormatter(lambda x, _: f"${x/1000:,.0f}K" if abs(x) >= 1000 else f"${x:,.0f}")

In [ ]:
# Path to the Excel file in Google Drive
url = '/content/drive/MyDrive/UPS Parcel Dataset - 2025.xlsx'

In [ ]:
# header=2 skips the two empty rows at the top of the sheet so the real column names are used
df_raw = pd.read_excel(url, sheet_name='Freight-Detail', header=2)
print(df_raw.shape)
display(df_raw.head())

## 2. Data cleaning & quality check
Excel columns come in with mixed types (numbers stored as text, dates as text). We convert them, drop columns that are entirely empty, and look at what's missing.

In [ ]:
df = df_raw.copy()

# 2a. Drop columns that are 100% empty (e.g. DeliveryDate, DaysInTransit, On-time are blank in this extract)
empty_cols = df.columns[df.isna().all()].tolist()
df = df.drop(columns=empty_cols)
print("Dropped empty columns:", empty_cols)

# 2b. Convert money / weight / dimension columns to numbers
num_cols = ['NetCharge', 'FreightCharge', 'FuelSurcharge', 'ResidentialServiceFee',
            'MiscCharges', 'EarlyAMDeliveryFee', 'ChargeableWeight', 'ManifestWeight',
            'DimensionalWeight', 'Height', 'Length', 'Width', 'ShippingZoneCd']
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce')

# 2c. Convert date columns
for c in ['ShipDate', 'InvDate']:
    df[c] = pd.to_datetime(df[c], errors='coerce')

# 2d. Remove exact duplicate rows (if any)
before = len(df)
df = df.drop_duplicates()
print(f"Duplicate rows removed: {before - len(df)}")
print("Final shape:", df.shape)

In [ ]:
# Quality check: % missing for columns that have any gaps
missing = (df.isna().mean() * 100).round(1)
display(missing[missing > 0].sort_values(ascending=False).to_frame('% missing'))

In [ ]:
# Quick look at the categories we will group by
for c in ['CarrierShortNm', 'MOT', 'ServiceTypeNm', 'DataSource', 'ShipToCountryCd']:
    print(df[c].value_counts().to_dict())

## 3. Derived metrics (feature engineering)
New columns that turn raw charges into business measures.

| New column | Meaning |
|---|---|
| `Lane` | Domestic (Plymouth MI → US) vs International import (Heverlee BE → AU/GB) |
| `Accessorials` | Everything on the bill that is not base freight or fuel |
| `Fuel_pct` | Fuel surcharge as % of base freight |
| `Cost_per_lb` | Net charge ÷ chargeable weight |
| `Invoice_lag_days` | Days between ship date and invoice date |
| `Has_resi_fee` | Whether a residential delivery fee was charged |
| `Weekday`, `Ship_Month` | Calendar breakdowns of the ship date |

In [ ]:
df['Lane'] = df['CarrierShortNm'].map({'UPS Parcel': 'Domestic',
                                         'UPS IMPORT': 'International'})
df['Accessorials']     = df['NetCharge'] - df['FreightCharge'] - df['FuelSurcharge']
df['Fuel_pct']         = np.where(df['FreightCharge'] > 0,
                                  df['FuelSurcharge'] / df['FreightCharge'], np.nan)
df['Cost_per_lb']      = df['NetCharge'] / df['ChargeableWeight'].replace(0, np.nan)
df['Invoice_lag_days'] = (df['InvDate'] - df['ShipDate']).dt.days
df['Has_resi_fee']     = df['ResidentialServiceFee'] > 0
df['Weekday']          = df['ShipDate'].dt.day_name()
df['Ship_Month']       = df['ShipDate'].dt.to_period('M').dt.to_timestamp()

# Handy subsets
dom  = df[df['Lane'] == 'Domestic'].copy()
intl = df[df['Lane'] == 'International'].copy()

display(df[['Lane', 'NetCharge', 'FreightCharge', 'FuelSurcharge', 'Accessorials',
            'Fuel_pct', 'Cost_per_lb', 'Invoice_lag_days', 'Weekday']].head())

## 4. KPI summary

In [ ]:
total = df['NetCharge'].sum()
kpi = df.groupby('Lane').agg(Shipments=('NetCharge', 'size'),
                             Spend=('NetCharge', 'sum'),
                             Avg_cost=('NetCharge', 'mean'),
                             Avg_weight_lb=('ChargeableWeight', 'mean'))
kpi['Share_of_spend'] = kpi['Spend'] / total
kpi.loc['Total'] = [len(df), total, df['NetCharge'].mean(),
                    df['ChargeableWeight'].mean(), 1.0]
display(kpi.style.format({'Shipments': '{:,.0f}', 'Spend': '${:,.0f}', 'Avg_cost': '${:,.2f}',
                          'Avg_weight_lb': '{:.1f}', 'Share_of_spend': '{:.0%}'}))
print(f"Period: {df.ShipDate.min():%d %b %Y} to {df.ShipDate.max():%d %b %Y}"
      "  (Dec-2024 and Dec-2025 are partial months)")

## 5. Visualizations
### 5.1 Monthly spend trend by lane
*Question: is spend growing, seasonal, or flat?*

In [ ]:
monthly = df.pivot_table(index='Ship_Month', columns='Lane', values='NetCharge', aggfunc='sum')
ax = monthly.plot(marker='o', lw=2)
ax.set_title('Monthly net spend by lane')
ax.yaxis.set_major_formatter(usd); ax.set_xlabel(''); ax.set_ylim(bottom=0)
plt.show()

full = monthly.iloc[1:-1]          # drop the two partial months
peak = full.sum(axis=1).idxmax()
print(f"DERIVED RESULT: Peak month = {peak:%b %Y} (${full.sum(axis=1).max():,.0f}). "
      f"Average full month = ${full.sum(axis=1).mean():,.0f}.")

### 5.2 Spend by mode & service
*Question: which services account for most of the spend?*

In [ ]:
svc = df.groupby(['MOT', 'ServiceTypeNm'])['NetCharge'].sum().sort_values()
labels = [f"{m} · {s}" for m, s in svc.index]
ax = plt.barh(labels, svc.values, color=COLORS[0])[0].axes
for y, v in enumerate(svc.values):
    ax.text(v, y, f"  ${v:,.0f} ({v/total:.0%})", va='center', fontsize=9)
ax.set_title('Spend by mode of transport & service'); ax.xaxis.set_major_formatter(usd)
ax.set_xlim(0, svc.max() * 1.3)
plt.show()

top2 = svc.sort_values(ascending=False).head(2).sum() / total
print(f"DERIVED RESULT: The top 2 services account for {top2:.0%} of all spend.")

### 5.3 What makes up the bill: freight vs fuel vs accessorials
*Question: how much of each dollar is the actual freight, and how much is surcharges?*

In [ ]:
parts = df.groupby('Lane')[['FreightCharge', 'FuelSurcharge', 'Accessorials']].sum().clip(lower=0)
share = parts.div(parts.sum(axis=1), axis=0)
ax = share.plot(kind='barh', stacked=True, width=0.5, edgecolor='white')
for c in ax.containers:
    ax.bar_label(c, labels=[f"{w:.0%}" if w > 0.04 else '' for w in c.datavalues],
                 label_type='center', color='white', fontweight='bold')
ax.set_title('Composition of net charge by lane'); ax.set_ylabel('')
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.legend(['Base freight', 'Fuel surcharge', 'Accessorials / fees'],
          ncol=3, loc='upper center', bbox_to_anchor=(0.5, -0.12), frameon=False)
plt.show()

surch = (parts['FuelSurcharge'] + parts['Accessorials']).sum() / parts.values.sum()
print(f"DERIVED RESULT: {surch:.0%} of total spend (${(parts['FuelSurcharge'] + parts['Accessorials']).sum():,.0f}) "
      "is surcharges and fees on top of base freight.")

### 5.4 Fuel surcharge rate over time (derived: fuel ÷ base freight)
*Question: is the fuel surcharge creeping up, independent of volume?*

In [ ]:
fuel_m = (df[df['FreightCharge'] > 0]
          .groupby(['Ship_Month', 'Lane'])[['FuelSurcharge', 'FreightCharge']].sum())
fuel_m = (fuel_m['FuelSurcharge'] / fuel_m['FreightCharge']).unstack()
ax = fuel_m.plot(marker='o', lw=2)
ax.set_title('Fuel surcharge as % of base freight')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0)); ax.set_xlabel('')
plt.show()

for lane in fuel_m.columns:
    s = fuel_m[lane].dropna()
    print(f"DERIVED RESULT ({lane}): fuel rate ranged {s.min():.1%} to {s.max():.1%}; "
          f"{s.index[0]:%b %y} = {s.iloc[0]:.1%} vs {s.index[-1]:%b %y} = {s.iloc[-1]:.1%}.")

### 5.5 Domestic cost per lb by zone (derived: net charge ÷ chargeable weight)
*Question: how much does distance (zone) raise the unit cost?*

In [ ]:
z = dom.groupby('ShippingZoneCd').agg(cost_per_lb=('Cost_per_lb', 'mean'),
                                      avg_cost=('NetCharge', 'mean'),
                                      shipments=('NetCharge', 'size'))
z.index = z.index.astype(int).astype(str)
ax = z['cost_per_lb'].plot(kind='bar', color=COLORS[0], width=0.6, rot=0)
for x, (v, n) in enumerate(zip(z['cost_per_lb'], z['shipments'])):
    ax.text(x, v, f"${v:,.0f}/lb\nn={n}", ha='center', va='bottom', fontsize=9)
ax.set_title('Domestic Next Day Air: average cost per lb by UPS zone')
ax.set_xlabel('UPS zone (farther →)'); ax.yaxis.set_major_formatter(usd)
ax.set_ylim(0, z['cost_per_lb'].max() * 1.25)
plt.show()

print(f"DERIVED RESULT: Zone {z.index[-1]} costs {z['cost_per_lb'].iloc[-1] / z['cost_per_lb'].iloc[0]:.1f}x "
      f"per lb vs zone {z.index[0]}. Zones 104-105 carry "
      f"{z.loc[['104', '105'], 'shipments'].sum() / z['shipments'].sum():.0%} of domestic volume.")

### 5.6 Cost distribution by service (boxplot)
*Question: how variable is the cost per shipment, and are there outliers?*

In [ ]:
# Shipment lines only: 'Fees' rows (import duties/taxes) would distort the Worldwide Saver box
plot_df = df[(df['NetCharge'] > 0) & (df['MOT'] != 'Fees')]
order = plot_df.groupby('ServiceTypeNm')['NetCharge'].median().sort_values().index
ax = sns.boxplot(data=plot_df, x='NetCharge', y='ServiceTypeNm', order=order,
                 color=COLORS[0], fliersize=3, width=0.5)
ax.set_title('Distribution of net charge per shipment by service')
ax.set_ylabel(''); ax.xaxis.set_major_formatter(usd)
plt.show()

early = dom.groupby('ServiceTypeNm')['NetCharge'].mean()
print(f"DERIVED RESULT: Next Day Air Early AM averages ${early['Next Day Air Early AM']:,.0f} vs "
      f"${early['Next Day Air']:,.0f} for standard Next Day Air "
      f"(+{early['Next Day Air Early AM'] / early['Next Day Air'] - 1:.0%} premium).")

### 5.7 Destination concentration: Pareto curve
*Question: do a few customer addresses drive most of the domestic spend (80/20 rule)?*

In [ ]:
by_addr = dom.groupby('ShipToStreetAddr1')['NetCharge'].sum().sort_values(ascending=False)
cum = by_addr.cumsum() / by_addr.sum()
pct_addr = np.arange(1, len(cum) + 1) / len(cum)

fig, ax = plt.subplots()
ax.plot(pct_addr, cum.values, lw=2, color=COLORS[0])
ax.axhline(0.8, ls='--', color='grey', lw=1)
n80 = (cum < 0.8).sum() + 1
ax.axvline(n80 / len(cum), ls='--', color='grey', lw=1)
ax.annotate(f"{n80} of {len(cum)} addresses\n= 80% of spend",
            (n80 / len(cum), 0.8), xytext=(10, -40), textcoords='offset points')
ax.set_title('Cumulative share of domestic spend by destination address')
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_xlabel('Share of destination addresses (ranked by spend)'); ax.set_ylabel('Share of spend')
plt.show()

print(f"DERIVED RESULT: {n80 / len(cum):.0%} of addresses ({n80}) generate 80% of domestic spend. "
      f"Top address alone = {by_addr.iloc[0] / by_addr.sum():.0%}.")
display(by_addr.head(10).to_frame('Spend').style.format('${:,.0f}'))

### 5.8 Top destination states & residential-fee exposure
*Question: where does domestic spend go, and how often do we pay residential delivery fees?*

In [ ]:
st = dom.groupby('ShipToStateCd').agg(spend=('NetCharge', 'sum'),
                                     shipments=('NetCharge', 'size'),
                                     resi_rate=('Has_resi_fee', 'mean')).nlargest(10, 'spend')
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
st['spend'][::-1].plot(kind='barh', ax=axes[0], color=COLORS[0])
axes[0].set_title('Top 10 states by spend'); axes[0].xaxis.set_major_formatter(usd)
st['resi_rate'][::-1].plot(kind='barh', ax=axes[1], color=COLORS[1])
axes[1].set_title('% of shipments charged a residential fee')
axes[1].xaxis.set_major_formatter(mtick.PercentFormatter(1.0)); axes[1].set_xlim(0, 1)
for a in axes: a.set_ylabel('')
plt.tight_layout(); plt.show()

resi_total = dom['ResidentialServiceFee'].sum()
print(f"DERIVED RESULT: {dom['Has_resi_fee'].mean():.0%} of domestic shipments carried a residential fee, "
      f"costing ${resi_total:,.0f} in total.")

### 5.9 When do we ship? Weekday × month heatmap
*Question: are there day-of-week patterns that affect staffing or pickup scheduling?*

In [ ]:
days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
heat = (df.pivot_table(index='Weekday', columns='Ship_Month', values='NetCharge',
                       aggfunc='size', fill_value=0).reindex(days))
heat.columns = heat.columns.strftime('%b-%y')
plt.figure(figsize=(14, 4.5))
ax = sns.heatmap(heat, cmap='Blues', annot=True, fmt='d', linewidths=1, cbar_kws={'label': 'Shipments'})
ax.set_title('Shipments by weekday and month'); ax.set_xlabel(''); ax.set_ylabel('')
plt.show()

wd = df['Weekday'].value_counts(normalize=True).reindex(days)
print(f"DERIVED RESULT: Monday is the busiest day ({wd['Monday']:.0%} of shipments); "
      f"weekends account for {wd[['Saturday', 'Sunday']].sum():.0%}.")

### 5.10 Consolidation opportunity
*Question: how often do we send more than one Next Day Air box to the **same address on the same day**? Each extra box is a candidate for consolidation.*

In [ ]:
grp = dom.groupby(['ShipDate', 'ShipToStreetAddr1', 'ShipToPostalCd'])
dom['boxes_same_day'] = grp['NetCharge'].transform('size')
dom['extra_box'] = grp.cumcount() > 0          # 2nd, 3rd... box to the same place that day

cons = dom.groupby('Ship_Month').agg(extra_boxes=('extra_box', 'sum'),
                                     extra_spend=('NetCharge', lambda s: s[dom.loc[s.index, 'extra_box']].sum()))
cons.index = cons.index.strftime('%b-%y')     # text labels for a clean bar axis
ax = cons['extra_spend'].plot(kind='bar', color=COLORS[1], rot=0, width=0.6)
ax.set_title('Spend on extra same-day boxes to the same address (consolidation candidates)')
ax.yaxis.set_major_formatter(usd); ax.set_xlabel('')
plt.show()

n_extra, s_extra = dom['extra_box'].sum(), dom.loc[dom['extra_box'], 'NetCharge'].sum()
print(f"DERIVED RESULT: {n_extra} of {len(dom):,} domestic shipments ({n_extra/len(dom):.0%}) were an extra box "
      f"to an address that already got one that day: ${s_extra:,.0f} of spend.")
print("   Note: this is an UPPER BOUND. A consolidated box is heavier and still costs something, "
      "and some may be time-critical.")

### 5.11 International imports: landed cost per shipment (freight + fees)
*Question: what does each import to Australia / UK really cost once duties/fees are added?*

In [ ]:
im = intl.pivot_table(index='ShipToCountryCd', columns='MOT', values='NetCharge', aggfunc='sum').fillna(0)
im['Shipments'] = intl[intl['MOT'] == 'Air Freight'].groupby('ShipToCountryCd').size()
per = im[['Air Freight', 'Fees']].div(im['Shipments'], axis=0)

ax = per.plot(kind='bar', stacked=True, rot=0, width=0.5, edgecolor='white')
for i, tot in enumerate(per.sum(axis=1)):
    ax.text(i, tot, f"${tot:,.0f}", ha='center', va='bottom', fontweight='bold')
ax.set_title('Import landed cost per shipment (freight + fees)')
ax.yaxis.set_major_formatter(usd); ax.set_xlabel('Destination country')
plt.show()

au_lb = intl[(intl.MOT == 'Air Freight') & (intl.ShipToCountryCd == 'AU')]
print(f"DERIVED RESULT: Australia imports cost ${per.loc['AU'].sum():,.0f} per shipment "
      f"(${au_lb['Cost_per_lb'].mean():,.0f}/lb freight) vs ${per.loc['GB'].sum():,.0f} for the UK. "
      f"AU = {im.loc['AU', ['Air Freight', 'Fees']].sum() / total:.0%} of total company spend.")

### 5.12 Billing lag: days from ship date to invoice
*Question: how quickly are shipments billed? (useful for accruals and cash planning)*

In [ ]:
ax = sns.histplot(data=df, x='Invoice_lag_days', hue='Lane', binwidth=1,
                  hue_order=['Domestic', 'International'],
                  multiple='layer', alpha=0.6, edgecolor='white')
ax.set_title('Days between ship date and invoice date'); ax.set_xlabel('Days')
plt.show()

lag = df.groupby('Lane')['Invoice_lag_days'].agg(['median', 'mean', 'max'])
display(lag.round(1))
print(f"DERIVED RESULT: Half of all shipments are invoiced within {df['Invoice_lag_days'].median():.0f} days; "
      f"{(df['Invoice_lag_days'] > 10).mean():.0%} take longer than 10 days.")

## 6. Summary of findings
Run all cells, then use the printed **DERIVED RESULT** lines to fill in this table for your report:

| # | Theme | Finding | So what? |
|---|---|---|---|
| 1 | Spend concentration | Two services ≈ 95% of spend | Negotiate rates on these first |
| 2 | Surcharges | Fuel + accessorials ≈ 1/4 of spend; fuel rate rose through the year | Fuel-cap / accessorial waivers in the contract |
| 3 | Distance | Zone 108 ≈ 2.6x the cost/lb of zone 102 | Ship far zones from a closer origin, or use ground when time allows |
| 4 | Customers | Small set of addresses = 80% of spend | Account-level shipping agreements |
| 5 | Consolidation | Repeat same-day boxes to one address | Batch orders per customer per day |
| 6 | Imports | Australia lane is the single most expensive lane | Review Worldwide Saver vs ocean/deferred options |

*Note: `DeliveryDate`, `DaysInTransit` and `On-time` are empty in this extract, so service-level (on-time) analysis isn't possible. The `FedEx $ (NoFuel)` column is about 10x lower than the UPS charge, so it does not look like a like-for-like rate and is left out.*